In [ ]:
###### Do the downstream analysis, add the yazi parsed meta information
################################## Test the bbknn result based on lung Fibroblasts lineage
import pickle 
import os
import re
import pandas as pd
import numpy as np
import sys
import scanpy as sc
# import scatlasvae  ## 
import scipy.sparse as sp
import scipy
import bbknn
import harmonypy as hm
import gc
import weakref
import resource

Fib_Endo_age_meta = pd.read_csv('external_inputs/Fib_Endo_age_gse_meta.csv',sep=',')
Fib_Endo_age_gse = pd.read_csv('external_inputs/Fib_Endo_age_gse.csv',sep=',')
Fib_Endo_age_meta = Fib_Endo_age_meta.rename(columns={
    'GSE_ID': 'GSE',
    'GSM_ID': 'GSM',
    'Species': 'species',
    'Tissue': 'tissue'
})



In [ ]:
######### 更换批次去除整合好后的表达数据，PCA 那个 h5ad 的表达数据也是经过normalized的，只能从 pickle 文件中获取
pickle_file = '/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2.pickle'
with open(pickle_file, 'rb') as f:
    adata_list_merge = pickle.load(f)   #  CustomUnpickler(f).load()


keep_genes_df = pd.read_csv('external_inputs/FibAtlas_gene.csv',sep=',')
keep_genes = keep_genes_df['gene'].to_list()
# 步骤3: 重新加载并过滤
for i, adata in enumerate(adata_list_merge):
    # 只保留高频基因
    genes_to_keep = [g for g in keep_genes if g in adata.var_names]
    adata_subset = adata[:, genes_to_keep]
    adata_list_merge[i] = adata_subset
    print(f"Dataset {i+1} filtered: {adata_subset.shape}")
    # 释放原对象
    del adata
    del adata_subset
    # gc.collect()


gc.collect()
adata = sc.concat(adata_list_merge, join='outer')
cellid = adata.obs_names
cellid_df = pd.DataFrame(cellid, columns=['cell_id'])
cellid_df.to_csv('external_inputs/FibAtlas_cellid.csv',sep=',',index=False)
cellid = pd.read_csv('external_inputs/FibAtlas_cellid.csv',sep=',')

adata = adata[cellid, :]
metadata = pd.read_csv('external_inputs/Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn_meta.tsv',sep='\t',index_col=0)  # 存的时候存了 index
metadata = metadata.loc[cellid]


In [ ]:
cellid_df = pd.DataFrame(cellid, columns=['cell_id'])
cellid_df.to_csv('external_inputs/EndoAtlas_cellid.csv',sep=',',index=False)


In [ ]:
# adata = sc.read_h5ad()
adata.obs['Age_group']  = adata.obs['age'].apply(assign_age_group)
adata_age  = adata[~np.array(list(map(lambda x: isstr(x),adata.obs['age'])))]
adata_age.obs['age'] = adata_age.obs['age'].astype(float) 
adata_age.obs['Age'] = adata_age.obs['age']
adata_age


In [ ]:
obs_df = adata_age.obs
merged_obs = obs_df.merge(
    Fib_Endo_age_meta, 
    on=['GSE', 'GSM'],  # 根据这两列进行匹配
    how='left',         # 左连接，保留 adata_age.obs 的所有行
    suffixes=('', '_1')  # 如果有重复列名，添加后缀
)

merged_obs.index = obs_df.index
adata_age.obs = merged_obs


In [ ]:
#### 在Fibro/Endo atlas 中添加信息
obs_df = adata.obs
merged_obs = obs_df.merge(
    Fib_Endo_age_meta, 
    on=['GSE', 'GSM'],  # 根据这两列进行匹配
    how='left',         # 左连接，保留 adata_age.obs 的所有行
    suffixes=('', '_1')  # 如果有重复列名，添加后缀
)

merged_obs.index = obs_df.index
merged_obs['Disease_Status_Reasoning'] = merged_obs['Disease_Status_Reasoning'].astype(str)
merged_obs['Disease_Status'] = merged_obs['Disease_Status'].astype(str)
merged_obs['tissue_1'] = merged_obs['tissue_1'].astype(str)
merged_obs['Disease_Status_Reasoning'] = merged_obs['Disease_Status_Reasoning'].replace({np.nan: 'None'})
merged_obs['Disease_Status'] = merged_obs['Disease_Status'].replace({np.nan: 'None'})
merged_obs['tissue_1'] = merged_obs['tissue_1'].replace({np.nan: 'None'})
adata.obs = merged_obs







In [ ]:
sorted(dir())


In [ ]:

############### 改成 4 个subtype后，产王老师说的那种 boxplot, 柱子窄一点的 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
# 设置Nature/Cell期刊风格的参数
plt.style.use('default')
rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.titlesize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
})
# 1. 提取数据框并添加Celltype信息
adata_meta = adata.obs[['leiden','Celltype_general','Celltype', 'Myfibro', 'eFibro', 'iFibro', 'qFibro', 'Myfibro1', 'eFibro1','iFibro1', 'qFibro1']].copy()
adata_meta['Celltype'] = adata_meta['Celltype'].astype(str) + '_C' + adata_meta['leiden'].astype(str)  
# 定义leiden到Celltype的映射字典
# 确保leiden列是字符串类型，方便处理
adata_meta['leiden'] = adata_meta['leiden'].astype(str)
# 3. 按照leiden分组计算平均值
# 注意：这里修正了列名大小写问题
feature_columns = ['Myfibro1', 'eFibro1', 'iFibro1', 'qFibro1']   #  'Myfibro1', 'eFibro1','iFibro1', 'qFibro1'
# 检查列名是否存在，确保大小写匹配
available_columns = [col for col in feature_columns if col in adata_meta.columns]
grouped_mean = adata_meta.groupby(['Celltype'])[available_columns].mean().reset_index()
grouped_mean['leiden'] = grouped_mean['Celltype'].str.split('_C').str[-1].astype(int) 
grouped_mean = grouped_mean.sort_values(['Celltype','leiden'])
print(grouped_mean[['leiden', 'Celltype']].to_string(index=False))
# 5. 可视化设置
n_leiden = len(grouped_mean)
print(f"\nLeiden群数: {n_leiden}")
# 设置子图布局：4行10列
n_rows =  6
n_cols =  8
n_subplots = n_rows * n_cols
# 如果leiden数量少于子图数量，调整布局
if n_leiden < n_subplots:
    n_cols = int(np.ceil(n_leiden / n_rows))
    n_subplots = n_rows * n_cols

colors = plt.cm.Set2(np.linspace(0, 1, len(available_columns)))
# 6. 创建图形
fig, axes = plt.subplots(n_rows, n_cols, figsize=(1.5*n_cols, 2*n_rows))
axes = axes.flatten()
# 设置图形标题
fig.suptitle('SS of subFibro markers(long_raw)',  # Signature score of subFibro(long list)
             fontsize=12, fontweight='bold', y=1.02)
# 7. 绘制每个leiden的柱状图（添加数值标签）
for i, row in enumerate(grouped_mean.itertuples()):
    if i >= len(axes):
        break
    leiden = row.leiden
    celltype = row.Celltype
    ax = axes[i]
    # 提取当前leiden的数据
    leiden_data = grouped_mean.loc[grouped_mean['leiden'] == leiden, available_columns].iloc[0]
    # 创建柱状图
    x_pos = np.arange(len(available_columns))
    bars = ax.bar(x_pos, leiden_data.values, color=colors, edgecolor='black', linewidth=0.5)
    # 添加数值标签到每个柱子顶部
    for bar_idx, (bar, value) in enumerate(zip(bars, leiden_data.values)):
        # 确定标签位置
        height = bar.get_height()
        # 如果值太小，将标签放在柱子内部顶部
        if height < grouped_mean[available_columns].values.max() * 0.05:
            # 小值：标签放在柱子内部，白色字体
            text_y = height * 0.8  # 放在柱子高度的80%位置
            color = 'black' # 小的值也还是 black
            va = 'bottom'
        else:
            # 正常值：标签放在柱子顶部上方
            text_y = height + grouped_mean[available_columns].values.max() * 0.01  # 稍微高于柱子
            color = 'black'
            va = 'bottom'
        
        # 格式化数值显示（根据数值大小选择格式）
        if abs(value) < 0.001:
            text = f'{value:.1e}'
        elif abs(value) < 0.01:
            text = f'{value:.4f}'
        elif abs(value) < 0.1:
            text = f'{value:.3f}'
        else:
            text = f'{value:.2f}'
        
        # 添加文本标签
        ax.text(
            bar_idx, text_y, text,
            ha='center', va=va,
            fontsize=6,  # 使用较小字体
            color=color,
            fontweight='bold' if color == 'white' else 'normal',
            rotation=0
        )
    # 设置子图标题（包含leiden和Celltype信息）
    # 缩短Celltype显示，只显示主要部分
    if len(celltype) > 20:
        # 如果Celltype太长，只显示前部分
        short_celltype = celltype # celltype[:17] + "..."
    else:
        short_celltype = celltype
    
    title_text =f'{short_celltype}'  # f'L{leiden}\n{short_celltype}'
    ax.set_title(title_text, fontsize=6, fontweight='bold', pad=3)
    # 隐藏x轴刻度
    ax.set_xticks([])
    # 设置y轴标签
    if i % n_cols == 0:  # 每行最左边的图显示y轴标签
        ax.set_ylabel('Signature Score', fontsize=8)
    
    # 设置y轴范围，为顶部标签留出空间
    y_max = grouped_mean[available_columns].values.max() * 1.15  # 增加15%的空间用于标签
    ax.set_ylim([0, y_max])
    # 添加网格线（仅y方向，更简洁）
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    # 移除顶部和右侧边框
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # 根据Celltype前缀添加背景色区分
    celltype_prefix =  celltype.split('_')[0]     # get_celltype_prefix(celltype) , 这个函数不在了
    if celltype_prefix == 'iFibro':  # iFibro
        ax.set_facecolor('#FFF5E6')  # 浅橙色背景
    elif celltype_prefix == 'eFibro':  # eFibro # 
        ax.set_facecolor('#E6F7FF')  # 浅蓝色背景
    elif celltype_prefix == 'Myfibro':  # Myfibro
        ax.set_facecolor('#F0E6FF')  # 浅紫色背景
    elif celltype_prefix == 'qFibro':  # qFibro
        ax.set_facecolor('#E6FFE6')  # 浅绿色背景

# 8. 隐藏多余的子图
for i in range(len(grouped_mean), len(axes)):  
    axes[i].axis('off')

# 9. 创建统一的图例
# 使用简化版特征名称作为图例标签
feature_short_names = []
for col in available_columns:
    if 'Myfibro1' in col:
        feature_short_names.append('Myfibro')
        # feature_short_names.append('Myfibro_l')
    elif 'iFibro1' in col:
        feature_short_names.append('iFibro')
    elif 'eFibro1' in col:
        feature_short_names.append('eFibro')
    elif 'qFibro1' in col:
        feature_short_names.append('qFibro')
    elif 'Myfibro' in col:
        feature_short_names.append('Myfibro')
    elif 'iFibro' in col:
        feature_short_names.append('iFibro')
    elif 'eFibro' in col:
        feature_short_names.append('eFibro')
    elif 'qFibro' in col:
        feature_short_names.append('qFibro')
    else:
        feature_short_names.append(col.replace('_4raw', ''))

# 创建自定义的图例句柄
legend_handles = [plt.Rectangle((0,0),1,1, color=colors[j], edgecolor='black', linewidth=0.5) 
                  for j in range(len(available_columns))]
# 创建Celltype分组图例
celltype_prefix_colors = {
    1: ('iFibro', '#FFF5E6'),
    2: ('eFibro', '#E6F7FF'),
    3: ('Myfibro', '#F0E6FF'),
    4: ('qFibro', '#E6FFE6')
}
# 创建Celltype图例句柄
celltype_handles = []
celltype_labels = []
for prefix in sorted(celltype_prefix_colors.keys()):
    label, color = celltype_prefix_colors[prefix]
    handle = plt.Rectangle((0,0),1,1, color=color, edgecolor='gray', linewidth=0.5, alpha=0.7)
    celltype_handles.append(handle)
    celltype_labels.append(label)

# Subtypes图例（左下区域）
fig.legend(legend_handles, feature_short_names, 
           loc='lower center', 
           bbox_to_anchor=(0.25, 0.01),  # 底部左侧
           ncol=len(available_columns),  # 水平排列
           fontsize=8,
           title='marker_list',
           title_fontsize=9,
           frameon=True,
           fancybox=True,
           shadow=False,
           framealpha=0.9)

# Celltype Groups图例（右下区域）
fig.legend(celltype_handles, celltype_labels,
           loc='lower center',
           bbox_to_anchor=(0.75, 0.01),  # 底部右侧
           ncol=4,  # 水平排列
           title='Celltype',
           title_fontsize=9,
           fontsize=8,
           frameon=True,
           fancybox=True,
           shadow=False,
           framealpha=0.9)

plt.tight_layout()
plt.subplots_adjust(top=0.95, bottom=0.1,left=0.05, right=0.9, hspace=0.4, wspace=0.3)
# 12. 保存图形
fig_path = 'external_inputs/OnlyFibro'
# plt.savefig(
#     fig_path+"leiden_subtype_Short_scores.pdf",
#     dpi=300,
#     bbox_inches='tight',
#     format='pdf')
fig.savefig(fig_path+'leiden_subtype_Long_scores_raw.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# 使用函数处理数据
genes_to_check = Fib_marker_all
meta_data = prepare_marker_data(
    adata, 
    genes_to_check, 
    celltype_col='Celltype', 
    output_path='/external_inputs/celltype_gene_exp.csv'
)

# 可选：查看前几行数据
print(meta_data.head())


In [ ]:
def visualize_meta_cells(adata, adata_meta, meta_info, 
                         leiden_col='leiden', sample_col='sample'):
    """
    可视化元细胞聚合效果
    """
    import matplotlib.pyplot as plt
    
    # 创建对比图
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. 原始细胞分布
    leiden_counts = adata.obs[leiden_col].value_counts().sort_index()
    axes[0, 0].bar(range(len(leiden_counts)), leiden_counts.values)
    axes[0, 0].set_xlabel('Leiden Cluster')
    axes[0, 0].set_ylabel('Number of Cells')
    axes[0, 0].set_title('Original Cells per Leiden Cluster')
    axes[0, 0].tick_params(axis='x', rotation=45)
    # 2. 元细胞分布
    meta_leiden_counts = adata_meta.obs[leiden_col].value_counts().sort_index()
    axes[0, 1].bar(range(len(meta_leiden_counts)), meta_leiden_counts.values)
    axes[0, 1].set_xlabel('Leiden Cluster')
    axes[0, 1].set_ylabel('Number of Meta Cells')
    axes[0, 1].set_title('Meta Cells per Leiden Cluster')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # 3. 压缩比例
    compression_ratios = []
    leiden_clusters = sorted(adata.obs[leiden_col].unique())
    
    for cluster in leiden_clusters:
        orig_count = (adata.obs[leiden_col] == cluster).sum()
        meta_count = (adata_meta.obs[leiden_col] == cluster).sum()
        if meta_count > 0:
            compression_ratios.append(orig_count / meta_count)
        else:
            compression_ratios.append(0)
    
    axes[0, 2].bar(range(len(compression_ratios)), compression_ratios)
    axes[0, 2].set_xlabel('Leiden Cluster')
    axes[0, 2].set_ylabel('Compression Ratio')
    axes[0, 2].set_title('Compression Ratio per Cluster')
    axes[0, 2].tick_params(axis='x', rotation=45)
    
    # 4. 样本分布比较
    sample_counts_orig = adata.obs[sample_col].value_counts().head(10)
    axes[1, 0].bar(range(len(sample_counts_orig)), sample_counts_orig.values)
    axes[1, 0].set_xlabel('Sample')
    axes[1, 0].set_ylabel('Number of Cells')
    axes[1, 0].set_title('Original Cells per Sample(Top 10)')
    axes[1, 0].tick_params(axis='x', rotation=45)
    
    # 5. 元细胞样本分布
    sample_counts_meta = adata_meta.obs[sample_col].value_counts().head(10)
    axes[1, 1].bar(range(len(sample_counts_meta)), sample_counts_meta.values)
    axes[1, 1].set_xlabel('Sample')
    axes[1, 1].set_ylabel('Number of Meta Cells') 
    axes[1, 1].set_title('Meta Cells per Sample(Top 10)')
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    # 6. 元细胞大小分布
    meta_sizes = adata_meta.obs['n_cells_in_meta'].value_counts().sort_index()
    axes[1, 2].bar(meta_sizes.index.astype(str), meta_sizes.values)
    axes[1, 2].set_xlabel('Meta Cell Size')
    axes[1, 2].set_ylabel('Number of Meta Cells')
    axes[1, 2].set_title('Distribution of Meta Cell Sizes')
    axes[1, 2].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    fig_dir ='external_inputs/trajectory'
    plt.savefig(fig_dir+"/Endo_meta_cell_summary.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False)
    plt.show()
    # 打印详细统计
    print("=" * 60)
    print("元细胞聚合详细统计")
    print("=" * 60)
    
    # 按leiden统计
    leiden_stats = []
    for cluster in leiden_clusters:
        orig_mask = adata.obs[leiden_col] == cluster
        meta_mask = adata_meta.obs[leiden_col] == cluster
        
        orig_count = orig_mask.sum()
        meta_count = meta_mask.sum()
        
        if meta_count > 0:
            ratio = orig_count / meta_count
            avg_size = adata_meta.obs.loc[meta_mask, 'n_cells_in_meta'].mean()
            
            leiden_stats.append({
                'Leiden': cluster,
                'Original_Cells': orig_count,
                'Meta_Cells': meta_count,
                'Compression_Ratio': f"{ratio:.2f}",
                'Avg_Meta_Size': f"{avg_size:.1f}"
            })
    
    leiden_stats_df = pd.DataFrame(leiden_stats)
    print("\n按Leiden聚类统计:")
    print(leiden_stats_df.to_string(index=False))
    
    # 按样本统计
    sample_stats = []
    top_samples = adata.obs[sample_col].value_counts().head(10).index
    
    for sample in top_samples:
        orig_mask = adata.obs[sample_col] == sample
        meta_mask = adata_meta.obs[sample_col] == sample
        orig_count = orig_mask.sum()
        meta_count = meta_mask.sum()
        
        if meta_count > 0:
            ratio = orig_count / meta_count
            
            sample_stats.append({
                'Sample': sample,
                'Original_Cells': orig_count,
                'Meta_Cells': meta_count,
                'Compression_Ratio': f"{ratio:.2f}"
            })
    
    sample_stats_df = pd.DataFrame(sample_stats)
    print("\n按样本统计 (Top 10):")
    print(sample_stats_df.to_string(index=False))


In [ ]:
############# 筛选 pan tissue 的数据产 meta-cell，然后重新整合，做trajectory analysis
# 筛选 Celltype 以 _pan 结尾的细胞
adata = adata[adata.obs['Celltype_general'].isin(['eFibro', 'Myfibro','qFibro','iFibro'])]
adata = adata[adata.obs['Celltype'].str.endswith('_pan')]  
### define the meta-cell generation function: 在 Saturn 中的  external_inputs/Trajectory_analysis.ipynb 脚本的 'create_meta_cells' &  'visualize_meta_cells' function 

########## 这里没有用 adata 的raw的表达数据来产 meta-cell, 做下整合看下
adata = sc.read_h5ad('/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyFib.h5ad')
adata = adata[adata.obs['Celltype'].str.endswith('_pan')] 
adata_meta, meta_info = create_meta_cells(
        adata, 
        n_cells_per_meta=30, 
        min_cells_to_keep=15,
        leiden_col='leiden', 
        sample_col='sample',
        use_rep='X_pca',  # 这里有个问题, 这个adata对象中的 X_pca 是之前的，这个adata只是merge(adata_list_merge),并没跑 PCA ？？？会不会影响
        random_state=42
 ) 
    





In [ ]:
######### 画 Endothelial Atlas 的 Celltype UMAP图，现在先不加那个圈圈，PDF也先不存
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
# 1. 基础检查
required_cols = ['leiden', 'Celltype_general', 'Celltype']
for col in required_cols:
    if col not in adata.obs.columns:
        raise ValueError(f"adata.obs 中缺少必要列: {col}")

# 确保 leiden 是字符串（Scanpy 推荐）
adata.obs['leiden'] = adata.obs['leiden'].astype(str)
adata.obs['Celltype'] = adata.obs['Celltype'].astype(str)
# 根据adata.obs['leiden'] 产生 leiden color list, 参考Nature系列的配色
# ================================
# 2. 构建 Leiden → 颜色映射
# ================================
leiden_categories = sorted(
    adata.obs['leiden'].unique(),
    key=lambda x: int(x)
)
if len(leiden_categories) != len(Nat_color_list):
    raise ValueError("Nat_color_list 数量必须等于 Leiden cluster 数量")

leiden_color_dict = dict(zip(leiden_categories, Nat_color_list))
# 告诉 scanpy 使用这套颜色
adata.uns['leiden_colors'] = [
    leiden_color_dict[c] for c in leiden_categories
]
# ================================
# 3. 构建 Celltype_C 列
# ================================

adata.obs['Celltype_C'] = (
    adata.obs['Celltype'] +
    "_C" +
    adata.obs['leiden']
)
# ================================
# 4. 构建 Celltype_C → 颜色映射（来自 Leiden）
# ================================
celltypeC_color_dict = {}
for leiden_id in leiden_categories:
    mask = adata.obs['leiden'] == leiden_id
    celltypeC_values = adata.obs.loc[mask, 'Celltype_C'].unique()
    for val in celltypeC_values:
        celltypeC_color_dict[val] = leiden_color_dict[leiden_id]

# ================================
# 5. 开始绘图
# ================================
plt.rcParams['figure.figsize'] = (13, 6.2)
fig, ax = plt.subplots()
# 关键：指定 palette=Nat_color_list
sc.pl.umap(
    adata,
    color='leiden',
    palette=Nat_color_list,
    ax=ax,
    show=False,
    legend_loc='on data',   # 不用scanpy默认legend
    s=10,
    alpha=0.8
)
ax.set_title('Endothelial Atlas', fontsize=16, fontweight='bold', pad=20)
# ================================
# 6. 去边框
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
# ----------------------
# 5. 添加带箭头的轴标签
# ----------------------
# 计算箭头长度（基于UMAP坐标范围，确保箭头比例协调）
x_range = ax.get_xlim()
y_range = ax.get_ylim()
arrow_length_x = (x_range[1] - x_range[0]) * 0.1  # 占X轴范围10%
arrow_length_y = (y_range[1] - y_range[0]) * 0.1  # 占Y轴范围10%
# 添加UMAP_1箭头（x轴0点向右）
ax.annotate(
    'UMAP_1',
    xy=(0, 0),  # 箭头起点（坐标轴0点）
    xytext=(arrow_length_x, 0),  # 箭头终点
    arrowprops=dict(
        arrowstyle='<-',  # 箭头样式
        color='black',    # 箭头颜色
        lw=1.5            # 箭头线宽
    ),
    fontsize=13,
    ha='left',  # 文本对齐方式
    va='center'
)
# 添加UMAP_2箭头（y轴0点向上，竖直文本）
ax.annotate(
    'UMAP_2',
    xy=(0, 0),  # 箭头起点（坐标轴0点）
    xytext=(0, arrow_length_y),  # 箭头终点
    arrowprops=dict(
        arrowstyle='<-',  # 箭头样式
        color='black',    # 箭头颜色
        lw=1.5            # 箭头线宽
    ),
    fontsize=13,
    ha='center',  # 文本对齐方式
    va='bottom',
    rotation=90   # 文本竖直排布
)

# ================================
# 8. 构建分组 legend（按 Celltype_general）
# ================================
general_groups =  ['SMC', 'EndoMT-I', 'CapEC', 'VenEC', 'ArtEC']  #    ['Peri', 'VenEC', 'LEC', 'EndoMT-II', 'EndoMT-I','ArtEC','CapEC','SMC']  ## LEC 就是 LymEC
col1_generals = [ 'VenEC', 'CapEC','SMC']
col2_generals = ['ArtEC','EndoMT-I']
def build_legend_handles(generals):
    handles = []
    labels = []
    for g in generals:
        subset = adata.obs[adata.obs['Celltype_general'] == g]
        unique_celltypeC = sorted(subset['Celltype_C'].unique())
        # 添加 general 标题（黑色）
        handles.append(mpatches.Patch(color='white', label=g))
        labels.append(g)
        # 添加每个 Celltype_C
        for ct in unique_celltypeC:
            handles.append(
                mpatches.Patch(
                    color=celltypeC_color_dict[ct],
                    label=ct
                )
            )
            labels.append("  " + ct)  
    return handles, labels


# 左 legend
handles1, labels1 = build_legend_handles(col1_generals)
legend1 = ax.legend(
    handles1,
    labels1,
    bbox_to_anchor=(1.01, 1.01),
    loc='upper left',
    frameon=False,
    fontsize=9
)
ax.add_artist(legend1)
# 右 legend
handles2, labels2 = build_legend_handles(col2_generals)
legend2 = ax.legend(
    handles2,
    labels2,
    bbox_to_anchor=(1.32, 1.01),
    loc='upper left',
    frameon=False,
    fontsize=9
)
# ----------------------
# 8. 计算每个 Celltype_general 对应的UMAP坐标边界（用于画圈）
# ----------------------
def get_cluster_bounds(adata, group_col):
    """计算每个分组的UMAP坐标边界（中心+宽高）"""
    bounds = {}
    for group in adata.obs[group_col].unique():
        # 获取该分组的UMAP坐标
        umap_coords = adata.obsm['X_umap'][adata.obs[group_col] == group]
        if len(umap_coords) == 0:
            continue
        # 计算中心坐标
        center_x = np.mean(umap_coords[:, 0])
        center_y = np.mean(umap_coords[:, 1])
        # 计算边界（用于确定圆圈大小）
        min_x, max_x = np.min(umap_coords[:, 0]), np.max(umap_coords[:, 0])
        min_y, max_y = np.min(umap_coords[:, 1]), np.max(umap_coords[:, 1])
        # 计算宽度和高度（扩大1.2倍，让圈更宽松）
        width = (max_x - min_x) * 1.2
        height = (max_y - min_y) * 1.2
        bounds[group] = {
            'center': (center_x, center_y),
            'width': width,
            'height': height
        }
    return bounds

###################### 这里计算边界，然后画圈的暂时先不要
# 计算每个Celltype_general的UMAP边界
cluster_bounds = get_cluster_bounds(adata, 'Celltype_general')

# ----------------------
# 9. 为每个Celltype_general添加弯曲虚线圆圈和文本标签
general_colors = {
    'Peri': '#1f77b4',       # 深海蓝（Cell经典主色）
    'VenEC': '#ff7f0e',    # 暖橙（Nature常用强调色）
    'EndoMT-II': '#2ca02c',# 森林绿（Science功能色）
    'CapEC': '#d62728',    # 猩红（顶级期刊高亮色）
    'LEC': '#9467bd',    # 紫罗兰（Cell子刊常用）
    'SMC': '#8c564b',      # 棕褐（Nature子刊纹理色）
    'ArtEC': '#e377c2',    # 蔷薇粉（Science子刊对比色）
    'EndoMT-I': '#7f7f7f'  # 石墨灰（期刊中性对比色）
}

# # ----------------------
# for general_type, color in general_colors.items():
#     if general_type not in cluster_bounds:
#         continue
        
#     # 获取该分组的边界信息
#     bounds = cluster_bounds[general_type]
#     center_x, center_y = bounds['center']
#     width = bounds['width']
#     height = bounds['height']
#     # 创建弯曲的椭圆虚线框（模拟弯曲圈）
#     ellipse = mpatches.Ellipse(
#         (center_x, center_y),
#         width=width,
#         height=height,
#         edgecolor=color,
#         facecolor='none',
#         linestyle='--',  # 虚线
#         linewidth=2,     # 线宽
#         alpha=0.8        # 透明度
#     )
#     ax.add_patch(ellipse)
#     # 添加文本标签（在圈的右侧/上方，避免遮挡）
#     # 计算标签位置（圈的右侧偏移一点）
#     label_x = center_x + width/2 + (x_range[1]-x_range[0])*0.02
#     label_y = center_y
#     # 添加文本
#     ax.text(
#         label_x, label_y,
#         general_type,
#         fontsize=12,
#         fontweight='bold',
#         color=color,
#         ha='left',
#         va='center',
#         bbox=dict(boxstyle="round,pad=0.3", facecolor='white', edgecolor=color, alpha=0.7)
#     )

# 10. 保存
plt.tight_layout()
fig_dir='external_inputs/OnlyEndo'
# plt.savefig(
#     fig_dir+"Endothelial_umap_leiden_Celltype.pdf",
#     dpi=300,
#     bbox_inches='tight',
#     format='pdf'
# )
plt.savefig(fig_dir+"/Endothelial_umap_leiden_Celltype_new.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False)
plt.show()



In [ ]:
############## Age 中数据的统计 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 设置图片风格
plt.style.use('default')
sns.set(font='Arial', style='whitegrid')

# -------------- 1. 处理 Disease_Status --------------
# 按 Disease_Status 分组统计：样本数（nunique）和细胞数（count）
disease_stats = merged_obs.groupby('Disease_Status').agg(
    n_samples=('sample', 'nunique'),
    n_cells=('sample', 'count')
).reset_index()

# 绘制双轴图：左轴细胞数，右轴样本数
fig, ax1 = plt.subplots(figsize=(3, 3))

# 细胞数柱状图
sns.barplot(
    data=disease_stats,
    x='Disease_Status',
    y='n_cells',
    ax=ax1,
    color='#66c2a5',
    alpha=0.8
)
ax1.set_xlabel('Disease Status', fontsize=12)
ax1.set_ylabel('N(cells)', fontsize=12, color='#2c7fb8')
ax1.tick_params(axis='y', labelcolor='#2c7fb8')

# 样本数折线图
ax2 = ax1.twinx()
sns.lineplot(
    data=disease_stats,
    x='Disease_Status',
    y='n_samples',
    ax=ax2,
    color='#fc8d62',
    marker='o',
    linewidth=2
)
ax2.set_ylabel('N(sample)', fontsize=12, color='#e41a1c')
ax2.tick_params(axis='y', labelcolor='#e41a1c')

plt.title('N(cells & sample) by Disease Status', fontsize=14)
plt.tight_layout()
plt.savefig('external_inputs/disease_sample_cell_distribution.png', dpi=300, bbox_inches='tight')
plt.close()

# -------------- 2. 处理 Age_group --------------
# 按 Age_group 分组统计：样本数（nunique）和细胞数（count）
age_stats = merged_obs.groupby('Age_group').agg(
    n_samples=('sample', 'nunique'),
    n_cells=('sample', 'count')
).reset_index()

# 按字母顺序排序（A-H）
age_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
age_stats['Age_group'] = pd.Categorical(age_stats['Age_group'], categories=age_order, ordered=True)
age_stats = age_stats.sort_values('Age_group')

# 绘制双轴图
fig, ax1 = plt.subplots(figsize=(5, 5))

# 细胞数柱状图
sns.barplot(
    data=age_stats,
    x='Age_group',
    y='n_cells',
    ax=ax1,
    color='#8da0cb',
    alpha=0.8
)
ax1.set_xlabel('Age Group', fontsize=12)
ax1.set_ylabel('N(cells)', fontsize=12, color='#2c7fb8')
ax1.tick_params(axis='y', labelcolor='#2c7fb8')

# 样本数折线图
ax2 = ax1.twinx()
sns.lineplot(
    data=age_stats,
    x='Age_group',
    y='n_samples',
    ax=ax2,
    color='#fc8d62',
    marker='o',
    linewidth=2
)
ax2.set_ylabel('N(sample)', fontsize=12, color='#e41a1c')
ax2.tick_params(axis='y', labelcolor='#e41a1c')

plt.title('N(cells & sample) by AgeGroup', fontsize=14)
plt.tight_layout()
plt.savefig('external_inputs/age_group_sample_cell_distribution.png', dpi=300, bbox_inches='tight')
plt.close()

# （可选）打印统计结果
print("=== Disease Status Stats ===")
print(disease_stats)
print("\n=== Age Group Stats ===")
print(age_stats)


In [ ]:
adata = sc.read_h5ad('/data/project/chenfeiwang/ytwang/Endo_human_EndoDEG_count_notAge_0.2_10X_tissueB_bbknn.h5ad')   # 可以试下 OnlyEndo & EndoTotal 以及 OnlyFibro & FibTotal, 这4个不同的单细胞 object


In [ ]:
################## Fibroblast & Endothelial 这边用 scib 包评估批次去除效果的代码, 抽样   #
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scib_metrics.benchmark import Benchmarker
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import pickle 
import os
import re
import sys
# import scatlasvae  ## 
import scipy.sparse as sp
import scipy
import bbknn
import harmonypy as hm
import gc
import weakref
import resource
# ============================================
# 1. 设置参数
# ============================================
N_SAMPLES = 200000      # 每次抽样的细胞数
N_ITERATIONS = 10       # 抽样次数
RANDOM_SEED = 42        # 随机种子
BATCH_KEY = 'tissue'    # 批次信息列名
LABEL_KEY = 'leiden'    # 细胞类型/聚类结果列名
EMBEDDING_KEY = 'X_pca' # 用于评估的嵌入
MIN_CELLS_PER_GROUP = 10  # 每个组合的最小细胞数阈值（低于此值认为是稀有组合）
STRATEGY = 'exclude'    # 处理策略: 'exclude'(排除稀有组合) 或 'include_all'(全部纳入)
MIN_CELLS_PER_COMBO = 5     # 每个 tissue-leiden 组合的最小细胞数要求
MIN_BATCHES = 2             # 最少需要多少个批次才能进行批次校正评估
MIN_LABELS = 2              # 最少需要多少个细胞类型才能进行生物保守性评估

# ============================================
# 2. 分析并过滤有效的组合
# ============================================
print("="*80)
print("步骤1: 分析 tissue-leiden 组合")
print("="*80)

# 创建组合列
#### 通过路径更换不同的 adata

adata = sc.read_h5ad('/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn.h5ad')
adata.obs['tissue_leiden'] = adata.obs[BATCH_KEY].astype(str) + '_' + adata.obs[LABEL_KEY].astype(str)

# 统计每个组合的细胞数
combo_counts = adata.obs['tissue_leiden'].value_counts()
print(f"\n总细胞数: {adata.n_obs:,}")
print(f"总组合数: {len(combo_counts)}")
print(f"组合细胞数范围: {combo_counts.min()} - {combo_counts.max()}")

# 识别有效组合（细胞数 >= MIN_CELLS_PER_COMBO）
valid_combos = combo_counts[combo_counts >= MIN_CELLS_PER_COMBO].index.tolist()
invalid_combos = combo_counts[combo_counts < MIN_CELLS_PER_COMBO].index.tolist()

print(f"\n有效组合 (≥{MIN_CELLS_PER_COMBO} cells): {len(valid_combos)}")
print(f"无效组合 (<{MIN_CELLS_PER_COMBO} cells): {len(invalid_combos)}")

# 显示无效组合
if invalid_combos:
    print(f"\n无效组合列表 (将被丢弃):")
    for combo in invalid_combos[:10]:
        print(f"  - {combo}: {combo_counts[combo]} 个细胞")
    if len(invalid_combos) > 10:
        print(f"  ... 还有 {len(invalid_combos)-10} 个")

# 过滤数据，只保留有效组合
adata_filtered = adata[adata.obs['tissue_leiden'].isin(valid_combos)].copy()
print(f"\n过滤后:")
print(f"  - 保留细胞数: {adata_filtered.n_obs:,} ({adata_filtered.n_obs/adata.n_obs*100:.1f}%)")
print(f"  - 保留组合数: {adata_filtered.obs['tissue_leiden'].nunique()}")
del adata
gc.collect()

# 检查过滤后是否还有足够的批次和细胞类型
remaining_batches = adata_filtered.obs[BATCH_KEY].nunique()
remaining_labels = adata_filtered.obs[LABEL_KEY].nunique()
print(f"  - 剩余批次: {remaining_batches}")
print(f"  - 剩余细胞类型: {remaining_labels}")

if remaining_batches < MIN_BATCHES:
    raise ValueError(f"过滤后批次数量不足 {MIN_BATCHES}，无法进行有效的批次校正评估")

if remaining_labels < MIN_LABELS:
    raise ValueError(f"过滤后细胞类型数量不足 {MIN_LABELS}，无法进行有效的生物保守性评估")

# ============================================
# 3. 分析每个批次中各个细胞类型的分布
# ============================================
print("\n" + "="*80)
print("步骤2: 分析每个批次中细胞类型的分布")
print("="*80)

# 创建透视表
combo_matrix = pd.crosstab(
    adata_filtered.obs[BATCH_KEY], 
    adata_filtered.obs[LABEL_KEY]
)
print(f"\n每个批次中各细胞类型的细胞数:")
print(combo_matrix)

# 找出每个批次中细胞数不足的细胞类型
insufficient_combos = []
for batch in combo_matrix.index:
    for label in combo_matrix.columns:
        cell_count = combo_matrix.loc[batch, label]
        if cell_count < MIN_CELLS_PER_COMBO:
            insufficient_combos.append(f"{batch}_{label}")
            print(f"  ⚠️ {batch} - {label}: {cell_count} 个细胞 (< {MIN_CELLS_PER_COMBO})")

if insufficient_combos:
    print(f"\n共有 {len(insufficient_combos)} 个组合细胞数不足，这些组合已被过滤")

# ============================================
# 4. 强制保证每个组合至少有5个细胞的分层抽样函数
# ============================================
def guaranteed_stratified_sampling(adata, batch_key, label_key, n_samples, n_iterations, 
                                     min_cells_per_combo=5, random_seed=42):
    """
    强制保证每个 tissue-leiden 组合在每次抽样中至少有 min_cells_per_combo 个细胞
    
    策略：
    1. 首先，从每个组合中固定抽取 min_cells_per_combo 个细胞
    2. 然后，从剩余的细胞池中按比例抽取剩余的细胞
    3. 确保总抽样数为 n_samples
    """
    np.random.seed(random_seed)
    # 创建组合列
    adata.obs['_batch_label'] = adata.obs[batch_key].astype(str) + '_' + adata.obs[label_key].astype(str)
    # 获取所有有效组合
    all_combos = adata.obs['_batch_label'].unique()
    combo_counts = adata.obs['_batch_label'].value_counts()
    # 验证每个组合的细胞数是否足够
    for combo in all_combos:
        if combo_counts[combo] < min_cells_per_combo:
            raise ValueError(f"组合 {combo} 只有 {combo_counts[combo]} 个细胞，少于要求的 {min_cells_per_combo}")
    
    n_combos = len(all_combos)
    print(f"\n强制抽样要求:")
    print(f"  - 有效组合数: {n_combos}")
    print(f"  - 每个组合最少抽取: {min_cells_per_combo} 个细胞")
    print(f"  - 固定抽取总数: {n_combos * min_cells_per_combo}")
    # 计算需要额外抽取的细胞数
    fixed_total = n_combos * min_cells_per_combo
    if fixed_total > n_samples:
        raise ValueError(f"固定抽取总数 ({fixed_total}) 超过单次抽样目标 ({n_samples})，请减少 MIN_CELLS_PER_COMBO 或增加 N_SAMPLES")
    
    flexible_total = n_samples - fixed_total
    print(f"  - 需要额外抽取: {flexible_total} 个细胞")
    # 计算每个组合在总数据中的比例（用于额外抽样）
    combo_props = combo_counts / len(adata)
    all_sample_indices = []
    for iteration in range(n_iterations):
        sample_indices = []
        # 第一部分：从每个组合中固定抽取 min_cells_per_combo 个细胞
        for combo in all_combos:
            combo_indices = np.where(adata.obs['_batch_label'] == combo)[0]
            # 无放回抽取固定数量
            fixed_samples = np.random.choice(combo_indices, min_cells_per_combo, replace=False)
            sample_indices.extend(fixed_samples)
        
        # 第二部分：按比例从每个组合中抽取额外的细胞
        # 注意：需要排除已经抽取的细胞
        used_indices = set(sample_indices)
        
        for combo in all_combos:
            # 该组合应该抽取的额外细胞数
            n_extra = int(round(flexible_total * combo_props[combo]))
            
            if n_extra > 0:
                # 获取该组合中未被使用的细胞
                combo_indices = np.where(adata.obs['_batch_label'] == combo)[0]
                available_indices = [idx for idx in combo_indices if idx not in used_indices]
                
                if len(available_indices) >= n_extra:
                    extra_samples = np.random.choice(available_indices, n_extra, replace=False)
                else:
                    # 如果可用细胞不足，允许有放回抽样
                    extra_samples = np.random.choice(combo_indices, n_extra, replace=True)
                
                sample_indices.extend(extra_samples)
                used_indices.update(extra_samples)
        
        # 如果抽样总数不足 n_samples，随机补充
        if len(sample_indices) < n_samples:
            remaining = n_samples - len(sample_indices)
            all_indices = np.arange(len(adata))
            available_indices = [idx for idx in all_indices if idx not in used_indices]
            if len(available_indices) >= remaining:
                extra = np.random.choice(available_indices, remaining, replace=False)
            else:
                extra = np.random.choice(all_indices, remaining, replace=True)
            sample_indices.extend(extra)
        elif len(sample_indices) > n_samples:
            sample_indices = np.random.choice(sample_indices, n_samples, replace=False)
        
        # 打乱顺序
        np.random.shuffle(sample_indices)
        all_sample_indices.append(np.array(sample_indices))
        
        # 验证每个组合的抽样数
        print(f"\n迭代 {iteration+1}:")
        print(f"  - 总抽样数: {len(sample_indices)}")
        
        # 验证每个组合的抽样数
        sample_combo_counts = pd.Series(adata.obs['_batch_label'].iloc[sample_indices]).value_counts()
        min_sampled = sample_combo_counts.min()
        print(f"  - 最小组合抽样数: {min_sampled}")
        
        if min_sampled < min_cells_per_combo:
            print(f"  ⚠️ 警告: 有组合抽样数不足 {min_cells_per_combo}")
    
    # 清理临时列
    del adata.obs['_batch_label']
    return all_sample_indices

# ============================================
# 5. 执行强制分层抽样
# ============================================
print("\n" + "="*80)
print("步骤3: 执行强制分层抽样")
print("="*80)

sampling_indices = guaranteed_stratified_sampling(
    adata_filtered,
    BATCH_KEY,
    LABEL_KEY,
    N_SAMPLES,
    N_ITERATIONS,
    MIN_CELLS_PER_COMBO,
    RANDOM_SEED
)

# ============================================
# 6. 增强的评估函数（详细的错误处理和验证）
# ============================================
def robust_evaluate_subsample(adata, indices, iteration_id, batch_key, label_key, 
                               embedding_key, min_cells_per_combo=5):
    """
    增强的评估函数，包含详细的验证和错误处理
    """
    # 创建子集
    adata_sub = adata[indices].copy()
    print(f"\n{'='*60}")
    print(f"迭代 {iteration_id + 1}/{N_ITERATIONS}")
    print(f"{'='*60}")
    print(f"抽样细胞数: {adata_sub.n_obs:,}")
    # 详细的验证
    print(f"\n验证抽样质量:")
    # 检查批次分布
    batch_counts = adata_sub.obs[batch_key].value_counts()
    print(f"  - 批次数: {len(batch_counts)}")
    print(f"  - 批次分布: {dict(batch_counts)}")
    # 检查细胞类型分布
    label_counts = adata_sub.obs[label_key].value_counts()
    print(f"  - 细胞类型数: {len(label_counts)}")
    print(f"  - 最小细胞类型大小: {label_counts.min()}")
    # 检查每个组合的细胞数
    adata_sub.obs['temp_combo'] = adata_sub.obs[batch_key].astype(str) + '_' + adata_sub.obs[label_key].astype(str)
    combo_counts_sub = adata_sub.obs['temp_combo'].value_counts()
    min_combo = combo_counts_sub.min()
    print(f"  - 最小组合细胞数: {min_combo}")
    # 验证是否满足最低要求
    if len(batch_counts) < 2:
        print(f"  ❌ 失败: 批次数 < 2，无法进行批次校正评估")
        return None
    
    if len(label_counts) < 2:
        print(f"  ❌ 失败: 细胞类型数 < 2，无法进行生物保守性评估")
        return None
    
    if min_combo < min_cells_per_combo:
        print(f"  ⚠️ 警告: 存在组合细胞数 < {min_cells_per_combo}，但继续尝试")
    
    # 检查 embedding 是否存在
    if embedding_key not in adata_sub.obsm:
        print(f"  ❌ 失败: {embedding_key} 不在 adata.obsm 中")
        return None
    
    # 确保至少还有足够的细胞
    min_required = 100  # scIB 通常需要至少100个细胞
    if adata_sub.n_obs < min_required:
        print(f"  ❌ 失败: 细胞数 {adata_sub.n_obs} < {min_required}")
        return None
    
    # 尝试运行 scIB
    try:
        print(f"\n运行 scIB 评估...")
        # 设置 Benchmarker，禁用一些可能有问题的指标
        bm = Benchmarker(
            adata_sub,
            batch_key=batch_key,
            label_key=label_key,
            embedding_obsm_keys=[embedding_key],
            n_jobs=4,  # 使用较少核心，避免资源竞争
            # 可选：跳过某些容易出错的指标
            # is_log1p_normed=True,  # 如果数据已经 log1p 标准化
        )
        # 运行评估
        bm.benchmark()
        # 获取结果
        results = bm.get_results(min_max_scale=False)
        if results is not None and len(results) > 0:
            #results.index = [f'bbknn_{iteration_id}']
            print(f"\n✅ 评估成功!")
            # 显示关键指标
            if 'Total' in results.columns:
                print(f"   Total Score: {results['Total'].values[0]:.4f}")
            if 'Bio conservation' in results.columns:
                print(f"   Bio conservation: {results['Bio conservation'].values[0]:.4f}")
            if 'Batch correction' in results.columns:
                print(f"   Batch correction: {results['Batch correction'].values[0]:.4f}")
            
            return results
        else:
            print(f"  ❌ 评估返回空结果")
            return None
            
    except Exception as e:
        print(f"\n❌ 评估失败:")
        print(f"   错误类型: {type(e).__name__}")
        print(f"   错误信息: {str(e)[:200]}")
        return None

# ============================================
# 7. 循环执行评估
# ============================================
print("\n" + "="*80)
print("步骤4: 执行 scIB 评估")
print("="*80)

evaluation_results = []
failed_iterations = []
evaluation_df = pd.DataFrame()

START_INDEX = 0   # 起始索引
END_INDEX = None  # 结束索引，None表示运行到末尾

total_iterations = min(N_ITERATIONS, len(sampling_indices))
start_iter = START_INDEX
end_iter = END_INDEX if END_INDEX is not None else total_iterations

print(f"将从索引 {start_iter} 运行到 {end_iter - 1} (共 {end_iter - start_iter} 次迭代)")

for i in tqdm(range(start_iter, end_iter), desc="评估进度", initial=start_iter, total=end_iter):
    result_df = robust_evaluate_subsample(
        adata_filtered,
        sampling_indices[i],
        i,
        BATCH_KEY,
        LABEL_KEY,
        EMBEDDING_KEY,
        MIN_CELLS_PER_COMBO
    )
    if result_df is not None:
        evaluation_results.append(result_df)
    else:
        failed_iterations.append(i)

# ============================================
# 8. 处理评估结果
# ============================================
print("\n" + "="*80)
print("步骤5: 处理评估结果")
print("="*80)

print(f"\n成功评估: {len(evaluation_results)}/{N_ITERATIONS}")
if failed_iterations:
    print(f"失败迭代: {failed_iterations}")

if len(evaluation_results) == 0:
    raise RuntimeError("所有评估都失败了！请检查：\n"
                      "1. 数据是否经过正确的预处理\n"
                      f"2. {EMBEDDING_KEY} 是否存在于 adata.obsm\n"
                      "3. tissue 和 leiden 列是否包含足够的类别\n"
                      "4. 是否安装了 scib-metrics (pip install scib-metrics)") 

# 合并结果
evaluation_df = pd.concat(evaluation_results, axis=0)
evaluation_df = evaluation_df.drop_duplicates()
evaluation_df = evaluation_df.drop(index='Metric Type').reset_index(drop=True)
evaluation_df['round'] = [f'bbknn_{i}' for i in range(len(evaluation_df))]

# ============================================
# 9. 保存结果
# ============================================
output_csv = f'external_inputs/Fib_total_bbknn_scvi_eval_min{MIN_CELLS_PER_COMBO}.csv'
evaluation_df.to_csv(output_csv,sep=',',index=False)
print(f"\n结果已保存到: {output_csv}")

# 计算统计信息
summary_stats = evaluation_df[['Bio conservation', 'Batch correction', 'Total']].agg(['mean', 'std']).round(4) 
summary_stats.to_csv(f'external_inputs/Fib_total_bbknn_scvi_eval_summary.csv',sep=',',index=False)
print(f"\n统计摘要:")
print(summary_stats)

# ============================================
# 10. 可视化 - 热力图
# ============================================
# 定义指标顺序
metric_order = ['Isolated labels', 'KMeans NMI', 'KMeans ARI', 'Silhouette label',
       'cLISI', 'Silhouette batch', 'iLISI', 'KBET', 'Graph connectivity',
       'PCR comparison', 'Batch correction', 'Bio conservation', 'Total']

existing_metrics = [m for m in metric_order if m in evaluation_df.columns]
if not existing_metrics:
    existing_metrics = ['Bio conservation', 'Batch correction', 'Total']




# 绘制热力图
heatmap_data = evaluation_df[existing_metrics]
heatmap_data_numeric = heatmap_data.apply(pd.to_numeric, errors='coerce')
heatmap_data_clean = heatmap_data_numeric.dropna()
heatmap_data_clean = heatmap_data_numeric.fillna(heatmap_data_numeric.mean())

# 设置图形大小
fig_height = max(6, len(heatmap_data_clean) * 0.4)
fig, ax = plt.subplots(figsize=(14, fig_height)) 
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.3f',
    cmap='RdYlGn',
    vmin=0, vmax=1,
    center=0.5,
    cbar_kws={'label': 'Score (higher is better)', 'shrink': 0.8},
    linewidths=0.5,
    square=False,
    ax=ax
)

ax.set_title(f'BBKNN Integration Benchmark (min {MIN_CELLS_PER_COMBO} cells per combo)\n'
             f'{len(evaluation_results)} successful subsamples, stratified by tissue & leiden', 
             fontsize=12, fontweight='bold')
ax.set_xlabel('Metrics', fontsize=11)
ax.set_ylabel('Subsample (bbknn_n)', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'external_inputs/Fib_total_bbknn_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================
# 11. 可视化 - 箱线图
# ============================================
fig, ax = plt.subplots(figsize=(10, 6))
key_metrics = ['Bio conservation', 'Batch correction', 'Total']
existing_key_metrics = [m for m in key_metrics if m in evaluation_df.columns]

if existing_key_metrics:
    plot_data = evaluation_df[existing_key_metrics].melt(var_name='Metric', value_name='Score')
    
    sns.boxplot(data=plot_data, x='Metric', y='Score', palette='Set2', ax=ax)
    sns.stripplot(data=plot_data, x='Metric', y='Score', color='black', alpha=0.5, size=6, ax=ax)
    
    ax.set_ylim(0, 1)
    ax.set_ylabel('Score (higher is better)', fontsize=12)
    ax.set_xlabel('Overall Metrics', fontsize=12)
    ax.set_title(f'BBKNN Performance Distribution (min {MIN_CELLS_PER_COMBO} cells per combo)',
                 fontsize=12, fontweight='bold')
    ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold (0.5)')
    ax.axhline(y=0.7, color='green', linestyle='--', alpha=0.5, label='Good (0.7)')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(f'external_inputs/Fib_total_bbknn_boxplot.png', dpi=300, bbox_inches='tight')
    plt.show()

# ============================================
# 12. 最终报告
# ============================================
print("\n" + "="*80)
print("最终评估报告")
print("="*80)
print(f"数据规模: {adata.n_obs:,} 细胞 × {adata.n_vars} 基因")
print(f"过滤后规模: {adata_filtered.n_obs:,} 细胞")
print(f"有效组合数: {len(valid_combos)}")
print(f"每个组合最小细胞数要求: {MIN_CELLS_PER_COMBO}")
print(f"成功评估次数: {len(evaluation_results)}/{N_ITERATIONS}")

if 'Total' in evaluation_df.columns:
    total_mean = evaluation_df['Total'].mean()
    total_std = evaluation_df['Total'].std()
    bio_mean = evaluation_df['Bio conservation'].mean() if 'Bio conservation' in evaluation_df.columns else None
    batch_mean = evaluation_df['Batch correction'].mean() if 'Batch correction' in evaluation_df.columns else None
    
    print(f"\n综合评分 (Total): {total_mean:.4f} ± {total_std:.4f}")
    if bio_mean:
        print(f"生物保守性 (Bio): {bio_mean:.4f} ± {evaluation_df['Bio conservation'].std():.4f}")
    if batch_mean:
        print(f"批次校正 (Batch): {batch_mean:.4f} ± {evaluation_df['Batch correction'].std():.4f}")
    
    print("\n" + "="*80)
    print("评估结论:")
    print("="*80)
    
    if total_mean >= 0.7:
        print("✅ 优秀: 批次去除效果优秀，可直接用于下游分析")
    elif total_mean >= 0.6:
        print("👍 良好: 批次去除效果良好，可以接受")
    elif total_mean >= 0.5:
        print("⚠️ 及格: 批次去除效果及格，建议结合UMAP可视化检查")
    else:
        print("❌ 不理想: 批次去除效果不理想，建议调整BBKNN参数")
    
    if bio_mean and batch_mean:
        if bio_mean < 0.5:
            print("\n⚠️ 建议: 生物保守性偏低，可能存在过度校正")
            print("   → 尝试减少 BBKNN 的 neighbors 参数")
        if batch_mean < 0.4:
            print("\n⚠️ 建议: 批次校正不足，批次效应残留明显")
            print("   → 尝试增加 BBKNN 的 neighbors 参数")

print("\n" + "="*80)
print("生成的文件:")
print("="*80)
print(f"1. {output_csv}")


In [ ]:
######    最原始的那一波代码，看下效果会不会好一点 
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scib_metrics.benchmark import Benchmarker
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ============================================
# 1. 设置参数
# ============================================
N_SAMPLES = 100000      # 每次抽样的细胞数
N_ITERATIONS = 10       # 抽样次数
RANDOM_SEED = 42        # 随机种子，保证可重复性
BATCH_KEY = 'tissue'    # 批次信息列名
LABEL_KEY = 'leiden'  # 细胞类型列名（请根据你的实际列名修改）
EMBEDDING_KEY = 'X_pca' # 用于评估的嵌入（应该是原始PCA，不是UMAP）

# ============================================
# 2. 获取总细胞数并生成抽样索引（无放回）
# ============================================
adata = sc.read_h5ad('/data/project/chenfeiwang/ytwang/Endo_human_EndoDEG_count_notAge_0.2_10X_tissueB_bbknn.h5ad')
np.random.seed(RANDOM_SEED)
total_cells = adata.n_obs
print(f"总细胞数: {total_cells}")

# 检查是否有足够的细胞进行无放回抽样
total_samples_needed = N_SAMPLES * N_ITERATIONS
if total_samples_needed > total_cells:
    print(f"警告: 需要 {total_samples_needed} 个细胞，但只有 {total_cells} 个细胞")
    print(f"将调整为: N_ITERATIONS = {total_cells // N_SAMPLES}")
    N_ITERATIONS = total_cells // N_SAMPLES
    total_samples_needed = N_SAMPLES * N_ITERATIONS

# 生成所有细胞的随机排列
all_indices = np.random.permutation(total_cells)

# 分割成N_ITERATIONS组，每组N_SAMPLES个细胞（无放回）
sampling_indices = []
for i in range(N_ITERATIONS):
    start_idx = i * N_SAMPLES
    end_idx = start_idx + N_SAMPLES
    sampling_indices.append(all_indices[start_idx:end_idx])

print(f"将进行 {N_ITERATIONS} 次无放回抽样，每次 {N_SAMPLES} 个细胞")

# ============================================
# 3. 定义单个抽样的评估函数
# ============================================
def evaluate_subsample(adata, indices, iteration_id, batch_key, label_key, embedding_key):
    """
    对单次抽样数据进行scIB评估
    """
    # 创建子集AnnData对象
    adata_sub = adata[indices].copy()
    
    print(f"\n迭代 {iteration_id + 1}/{N_ITERATIONS}: 抽样细胞数 {adata_sub.n_obs}")
    
    # 检查抽样后是否还有足够的批次和细胞类型
    n_batches = adata_sub.obs[batch_key].nunique()
    n_labels = adata_sub.obs[label_key].nunique()
    print(f"  - 包含 {n_batches} 个批次, {n_labels} 个细胞类型")
    
    # 如果某个批次或细胞类型数量太少，给出警告
    min_batch_size = adata_sub.obs[batch_key].value_counts().min()
    min_label_size = adata_sub.obs[label_key].value_counts().min()
    if min_batch_size < 10:
        print(f"  - 警告: 最小批次大小 = {min_batch_size} (可能影响评估稳定性)")
    if min_label_size < 10:
        print(f"  - 警告: 最小细胞类型大小 = {min_label_size} (可能影响评估稳定性)")
    try:
        # 初始化Benchmarker
        bm = Benchmarker(
            adata_sub,
            batch_key=batch_key,
            label_key=label_key,
            embedding_obsm_keys=[embedding_key],
            n_jobs=-1,  # 使用所有CPU核心，可根据需要调整
        )
        # 运行评估
        bm.benchmark()
        # 获取结果（不缩放）
        results = bm.get_results(min_max_scale=False)
        # 添加方法名称
        # results.index = [f'bbknn_{iteration_id}']
        
        return results
    
    except Exception as e:
        print(f"  错误: {e}")
        return None

# ============================================
# 4. 循环执行评估
# ============================================
evaluation_results = []

for i in tqdm(range(N_ITERATIONS), desc="评估进度"):
    result_df = evaluate_subsample(
        adata, 
        sampling_indices[i], 
        i,
        BATCH_KEY,
        LABEL_KEY,
        EMBEDDING_KEY
    )
    if result_df is not None:
        evaluation_results.append(result_df)

# 合并所有结果
if evaluation_results:
    evaluation_df = pd.concat(evaluation_results, axis=0)
else:
    raise RuntimeError("所有评估都失败了，请检查数据或参数设置")


evaluation_df = evaluation_df.drop_duplicates()
evaluation_df = evaluation_df.drop(index='Metric Type').reset_index(drop=True)
evaluation_df['round'] = [f'bbknn_{i}' for i in range(len(evaluation_df))]

# ============================================
# 5. 保存结果到CSV
# ============================================
output_csv = 'external_inputs/Fib_total_bbknn_scvi_eval_origin.csv'
evaluation_df.to_csv(output_csv,sep=',',index=False)
print(f"\n结果已保存到: {output_csv}")
print("\n评估结果汇总:")
print(evaluation_df)



# ============================================
# 6. 计算统计信息
# ============================================
# 计算均值和标准差
summary_stats = evaluation_df.agg(['mean', 'std']).round(4)
summary_stats.to_csv('external_inputs/Fib_total_bbknn_scvi_eval_origin_summary.csv')
print("\n统计摘要 (均值 ± 标准差):")
print(summary_stats)

# ============================================
# 7. 可视化 - 热力图
# ============================================
# 选择要展示的指标列（去掉非数值列）
metric_columns = [col for col in evaluation_df.columns if col not in ['Method']]

# 设置图形大小
fig_height = max(6, N_ITERATIONS * 0.4)
fig, ax = plt.subplots(figsize=(14, fig_height))

# 绘制热力图
# 注意：scIB指标通常范围是0-1，直接使用cmap='RdYlGn'
heatmap_data = evaluation_df[metric_columns]

# 如果有太多列，可以精简一下
if len(metric_columns) > 15:
    # 保留关键的评估指标
    important_metrics = [
        'Isolated labels', 'Leiden NMI', 'Leiden ARI', 
        'Silhouette label', 'cLISI', 'iLISI', 
        'KBET', 'Graph connectivity', 'PCR comparison',
        'Bio conservation', 'Batch correction', 'Total'
    ]
    # 只保留存在的指标
    existing_metrics = [m for m in important_metrics if m in heatmap_data.columns]
    heatmap_data = heatmap_data[existing_metrics]

# 绘制热力图
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.3f',
    cmap='RdYlGn',
    vmin=0, vmax=1,
    center=0.5,
    cbar_kws={'label': 'Score (higher is better)', 'shrink': 0.8},
    linewidths=0.5,
    square=False,
    ax=ax
)

ax.set_title(f'BBKNN Integration Benchmark (n={N_ITERATIONS} subsamples, {N_SAMPLES} cells each)', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Metrics', fontsize=12)
ax.set_ylabel('Subsample', fontsize=12)

plt.tight_layout()
plt.savefig('Endo_bbknn_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================
# 8. 可视化 - 箱线图（展示指标分布）
# ============================================
# 选择关键指标绘制箱线图
key_metrics = ['Bio conservation', 'Batch correction', 'Total']
existing_key_metrics = [m for m in key_metrics if m in evaluation_df.columns]

if existing_key_metrics:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # 准备数据
    plot_data = evaluation_df[existing_key_metrics].melt(var_name='Metric', value_name='Score')
    
    # 绘制箱线图
    sns.boxplot(data=plot_data, x='Metric', y='Score', palette='Set2', ax=ax)
    sns.stripplot(data=plot_data, x='Metric', y='Score', color='black', alpha=0.5, size=5, ax=ax)
    
    ax.set_ylim(0, 1)
    ax.set_ylabel('Score (higher is better)', fontsize=12)
    ax.set_xlabel('Overall Metrics', fontsize=12)
    ax.set_title(f'BBKNN Integration Performance Distribution (n={N_ITERATIONS} subsamples)', 
                 fontsize=14, fontweight='bold')
    ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold (0.5)')
    ax.axhline(y=0.7, color='green', linestyle='--', alpha=0.5, label='Good (0.7)')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig('Endo_bbknn_boxplot.png', dpi=300, bbox_inches='tight')
    plt.show()

# ============================================
# 9. 可视化 - 折线图（展示各次抽样的波动）
# ============================================
if existing_key_metrics:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for metric in existing_key_metrics:
        ax.plot(range(len(evaluation_df)), evaluation_df[metric].values, 
                marker='o', label=metric, linewidth=2, markersize=8)
    
    ax.set_xlabel('Subsample Index', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('BBKNN Performance Across Subsamples', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.set_xticks(range(len(evaluation_df)))
    ax.set_xticklabels([f'bbknn_{i}' for i in range(len(evaluation_df))], rotation=45)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
    ax.axhline(y=0.7, color='green', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig('Endo_bbknn_trend.png', dpi=300, bbox_inches='tight')
    plt.show()

# ============================================
# 10. 输出最终报告
# ============================================
print("\n" + "="*60)
print("评估完成！生成的文件:")
print("="*60)
print(f"1. {output_csv} - 详细评估结果")
print("2. Endo_bbknn_evaluation_summary.csv - 统计摘要")
print("3. Endo_bbknn_heatmap.png - 热力图")
print("4. Endo_bbknn_boxplot.png - 箱线图")
print("5. Endo_bbknn_trend.png - 趋势图")
print("="*60)

# 输出关键统计信息
print("\n关键指标统计 (均值 ± 标准差):")
for metric in existing_key_metrics:
    mean_val = evaluation_df[metric].mean()
    std_val = evaluation_df[metric].std()
    print(f"  {metric}: {mean_val:.4f} ± {std_val:.4f}")

# 判断批次去除效果
bio_mean = evaluation_df['Bio conservation'].mean() if 'Bio conservation' in evaluation_df.columns else None
batch_mean = evaluation_df['Batch correction'].mean() if 'Batch correction' in evaluation_df.columns else None
total_mean = evaluation_df['Total'].mean() if 'Total' in evaluation_df.columns else None

print("\n" + "="*60)
print("批次去除效果评估结论:")
print("="*60)
if total_mean:
    if total_mean >= 0.7:
        print(f"✅ 优秀: Total Score = {total_mean:.4f} ≥ 0.7")
        print("   批次去除效果优秀，可直接用于下游分析")
    elif total_mean >= 0.6:
        print(f"👍 良好: Total Score = {total_mean:.4f} ≥ 0.6")
        print("   批次去除效果良好，可以接受")
    elif total_mean >= 0.5:
        print(f"⚠️ 及格: Total Score = {total_mean:.4f} ≥ 0.5")
        print("   批次去除效果及格，建议结合UMAP可视化检查")
    else:
        print(f"❌ 不理想: Total Score = {total_mean:.4f} < 0.5")
        print("   批次去除效果不理想，建议调整BBKNN参数或尝试其他方法")
    
    if bio_mean and batch_mean:
        print(f"\n详细分解:")
        print(f"  - 生物保守性 (Bio conservation): {bio_mean:.4f}")
        print(f"  - 批次校正 (Batch correction): {batch_mean:.4f}")





In [ ]:
################## 画统计图   #############
h5ad_path = ['/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyFib_age.h5ad',
'/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyFib_age_normal.h5ad',
'/data/project/chenfeiwang/ytwang/Endo_human_EndoDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyEndo_age.h5ad',
'/data/project/chenfeiwang/ytwang/Endo_human_EndoDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyEndo_age_normal.h5ad'
    
]
adata = sc.read_h5ad(h5ad_path[1])
Fib_meta_data = adata.obs
age_gsm_df = Fib_meta_data[["Age_group", "GSM"]].copy()
age_gsm_df.rename(columns={'Age_group': 'age_group'}, inplace=True)

# 统计细胞数和原始GSM数
age_cell_count = age_gsm_df.groupby("age_group").size().reset_index(name="cell_count")
age_gsm_count = age_gsm_df.groupby("age_group")["GSM"].nunique().reset_index(name="gsm_count")

# 合并数据并处理
age_stats = pd.merge(age_cell_count, age_gsm_count, on="age_group", how="outer")
age_stats = age_stats.dropna(subset=["cell_count", "gsm_count"]).reset_index(drop=True)
age_stats["cell_count"] = age_stats["cell_count"].astype(int)
age_stats["gsm_count"] = age_stats["gsm_count"].astype(int)

# 设置age_group顺序（与实际分组一致）
custom_age_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G','H']  # H 只有1个样本，143 cells，去掉
age_stats = age_stats[age_stats["age_group"].isin(custom_age_order)].copy()
age_stats["age_group"] = pd.Categorical(age_stats["age_group"], categories=custom_age_order, ordered=True)
age_stats = age_stats.sort_values("age_group").reset_index(drop=True)

# --------------------------
# 核心1：计算放大500倍的GSM负向高度（柱子高度=实际GSM数×500，负向显示）
# --------------------------
zoom_factor = 500  # 固定放大500倍
age_stats["gsm_count_neg_zoom"] = -age_stats["gsm_count"] * zoom_factor  # 放大后的负向高度

# 打印统计结果（确认实际GSM数与放大后高度）
print("实际GSM数与放大后柱子高度：")
print(age_stats[["age_group", "gsm_count", "gsm_count_neg_zoom"]])
print(f"\n放大倍数：{zoom_factor}倍（GSM柱子高度=实际GSM数×{zoom_factor}）")


# --------------------------
# 步骤2：绘制柱状图（核心2：自定义y轴负向刻度标签）
# --------------------------
if not age_stats.empty:
    plt.figure(figsize=(8.8, 5))
    # 定义颜色
    cell_color = '#1F77B4'  # colors = ['#1F77B4', '#FF7F0E']
    gsm_color = '#FF7F0E'
    # 1. 绘制细胞数（向上，原始高度，不变）
    bars_cell = plt.bar(
        x=age_stats["age_group"],
        height=age_stats["cell_count"],
        color=cell_color,
        alpha=0.8,
        label="Cell Number"
    )
    # 2. 绘制GSM数（向下，放大500倍的高度）
    bars_gsm = plt.bar(
        x=age_stats["age_group"],
        height=age_stats["gsm_count_neg_zoom"],  # 用放大后的高度
        color=gsm_color,
        alpha=0.8,
        label="Sample Number"
    )
    ax = plt.gca()  # 获取当前坐标轴对象
    y_ticks = ax.get_yticks()  # 所有y轴刻度值（如[-6500, -3250, 0, 29027, 58055, ...]）
    # 自定义刻度标签：负向刻度÷-500（还原为实际GSM数），正向刻度保持不变
    custom_y_labels = []
    for tick in y_ticks:
        if tick < 0:  # 负向刻度（对应GSM）：刻度值 ÷ (-zoom_factor) = 实际GSM数
            actual_gsm = int(tick / (-zoom_factor))  # 如-6500 ÷ -500 = 13
            custom_y_labels.append(str(actual_gsm))  # 显示实际GSM数（如"13"）
        elif tick == 0:  # 0刻度：保持"0"
            custom_y_labels.append("0")
        else:  # 正向刻度（对应细胞数）：保持原始数值（可加千分位）
            custom_y_labels.append(f"{int(tick):,}")  # 如"29,027"
    # 应用自定义刻度标签（关键！让y轴负向显示实际GSM数）
    ax.set_yticklabels(custom_y_labels, fontsize=9)
    # --------------------------
    # 标注与美化（标注均显示实际数量，无放大）
    # --------------------------
    # 细胞数标注（显示实际数量，千分位）
    for bar, count in zip(bars_cell, age_stats["cell_count"]):
        plt.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + age_stats["cell_count"].max() * 0.01,
            s=f"{count:,}",
            ha="center", va="bottom",
            fontsize=9,
            fontweight="bold"
        )
    # GSM标注（显示实际数量，无需放大）
    for bar, count in zip(bars_gsm, age_stats["gsm_count"]):
        plt.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() - (age_stats["gsm_count"].max() * zoom_factor) * 0.02,  # 基于放大后高度的偏移
            s=f"{count}",  # 显示实际GSM数（如13）
            ha="center", va="top",
            fontsize=9,
            fontweight="bold",
            color=gsm_color
        )
    plt.xlabel("Age Group", fontsize=12, labelpad=10)
    plt.ylabel("Count", 
               fontsize=11, labelpad=10)
    plt.title(f"Fibro- Age, Healthy", 
              fontsize=14, pad=20)
    # 图例和0刻度线（加粗0刻度，区分上下区域）
    plt.legend(loc="upper center", fontsize=10, bbox_to_anchor=(0.5, -0.08), ncol=2)
    plt.axhline(y=0, color="black", linestyle="-", linewidth=2)
    # 调整布局，避免标签截断
    plt.tight_layout
    plt.savefig('/external_inputs/FibroAgeNormal_cellsample_agegroup.png', dpi=300, bbox_inches='tight')
    plt.savefig('external_inputs/FibroAgeNormal_cellsample_agegroup.pdf', format='pdf', bbox_inches='tight')
    plt.show()
else:
    print("无可用数据用于绘图，请检查custom_age_order是否与age_group匹配！")
    




In [ ]:
###############  画 Celltype proportion trend 的 function  ####
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd
import numpy as np

def plot_box_by_celltype_general(adata, age_group, age_order, celltype_general_col, celltype_col,lineage=None, Type=None):
    """
    为每个Celltype_general创建多个子图，每个子图对应一个具体的Celltype
    Parameters:
    -----------
    adata : AnnData
        单细胞数据对象
    age_group : str
        年龄组列名
    age_order : list
        年龄组的顺序
    celltype_general_col : str
        通用细胞类型列名
    celltype_col : str
        具体细胞类型列名
    """
    # 获取每个Celltype_general对应的具体Celltype列表
    celltype_mapping = adata.obs[[celltype_col, celltype_general_col]].drop_duplicates()
    # 按Celltype_general分组
    general_types = celltype_mapping[celltype_general_col].unique()
    for general_type in general_types:
        # 获取当前general_type对应的所有具体celltype
        specific_celltypes = celltype_mapping[celltype_mapping[celltype_general_col] == general_type][celltype_col].tolist()
        # 计算每个具体celltype在每个样本中的比例
        celltype_proportions = []
        for celltype in specific_celltypes:
            # 计算当前celltype在每个个体中的数量
            cell_counts = adata.obs[adata.obs[celltype_col] == celltype].groupby('sample').size().reset_index(name='count') 
            # 计算每个个体的总细胞数
            total_counts = adata.obs.groupby('sample').size().reset_index(name='total_count')
            # 过滤掉adata中没有的个体
            total_counts = total_counts[total_counts['sample'].isin(adata.obs['sample'])]
            # 合并并计算百分比
            cell_counts = cell_counts.merge(total_counts, on='sample', how='right').fillna(0)
            cell_counts['percentage'] = cell_counts['count'] / cell_counts['total_count'] * 100
            cell_counts['Celltype'] = celltype
            # 添加年龄组信息
            cell_counts = cell_counts.merge(
                adata.obs[[age_group, 'sample']].drop_duplicates(), 
                on='sample', 
                how='left'
            )
            cell_counts[age_group] = pd.Categorical(
                cell_counts[age_group], 
                categories=age_order, 
                ordered=True
            )
            celltype_proportions.append(cell_counts)
        # 合并所有celltype的数据
        all_data = pd.concat(celltype_proportions, ignore_index=True)
        # 确定子图布局
        n_celltypes = len(specific_celltypes)
        n_cols = min(3, n_celltypes)  # 最多3列
        n_rows = (n_celltypes + n_cols - 1) // n_cols
        # 创建子图
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
        if n_rows == 1 and n_cols == 1:
            axes = np.array([axes])
        axes = axes.flatten()
        # 为每个celltype绘制箱线图
        for idx, celltype in enumerate(specific_celltypes):
            ax = axes[idx]
            data_subset = all_data[all_data['Celltype'] == celltype]
            # 过滤掉没有数据的年龄组
            valid_groups = [group for group in age_order 
                          if len(data_subset[data_subset[age_group] == group]) > 0]
            if len(valid_groups) == 0:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
                ax.set_title(f"{celltype}\n(No data)", fontsize=10)
                continue
            # 计算统计值
            groups = [data_subset[data_subset[age_group] == group]['percentage'] 
                     for group in valid_groups if len(data_subset[data_subset[age_group] == group]) > 0]
            
            if len(groups) > 1:  # 至少需要两个组才能计算统计量
                try:
                    # Kruskal-Wallis H检验
                    h_stat, p_val = stats.kruskal(*groups)
                    
                    # 计算Eta Squared (η²)
                    n = sum([len(group) for group in groups])
                    eta_squared = h_stat**2 / (h_stat**2 + n - len(groups))
                    
                    # 根据效应量和p值确定显著性标记
                    if eta_squared < 0.01:
                        significance = " (ns)"
                    else:
                        if p_val <= 0.001:
                            significance = " (***)"
                        elif p_val <= 0.01:
                            significance = " (**)"
                        elif p_val <= 0.05:
                            significance = " (*)"
                        else:
                            significance = " (ns)"
                except:
                    p_val = 1.0
                    eta_squared = 0.0
                    significance = " (ns)"
            else:
                p_val = 1.0
                eta_squared = 0.0
                significance = " (ns)"
            
            # 绘制箱线图
            sns.boxplot(
                data=data_subset, 
                x=age_group, 
                y="percentage",
                order=valid_groups,
                palette="Set2", 
                width=0.6, 
                showfliers=False,
                ax=ax
            )
            # 设置标题和标签
            ax.set_title(f"{celltype}{significance}", fontsize=10, fontweight="bold")
            ax.set_xlabel("Age Group", fontsize=9)
            ax.set_ylabel("Percentage (%)", fontsize=9)
            ax.tick_params(axis='x', rotation=45)
            # 添加统计信息
            if len(groups) > 1:
                y_max = data_subset['percentage'].max() if len(data_subset) > 0 else 0
                ax.text(0.5, 0.95, f"p={p_val:.2e}\nη²={eta_squared:.3f}", 
                       ha='center', va='top', transform=ax.transAxes,
                       fontsize=8, color='black', weight='bold')
        # 隐藏多余的子图
        for idx in range(len(specific_celltypes), len(axes)):
            axes[idx].set_visible(False)
        # 设置主标题
        plt.suptitle(f"{general_type} - Cell Type Proportions Across Age Groups", 
                    fontsize=14, fontweight="bold", y=1.02)
        plt.tight_layout()
        fig_path = 'external_inputs/Celltype'+ Type+'/' +lineage+'/' 
        plt.savefig(
        fig_path+general_type +"_agegtrend.pdf",
        dpi=300, bbox_inches='tight',format='pdf')
        plt.savefig(fig_path+general_type +"_agegtrend.png", dpi=300, bbox_inches="tight")
        plt.show()



In [ ]:
def plot_box(adata, age_group, age_order, cell_subtype,lineage=None,Type=None):
    # 计算细胞类型在每个个体中的比例
    cell_counts = adata.obs.groupby(['sample', cell_subtype]).size().reset_index(name='count')
    total_counts = adata.obs.groupby('sample').size().reset_index(name='total_count')
    # total_counts出现了一些adata中没有的个体，导致NaN,把他们去掉
    total_counts = total_counts[total_counts['sample'].isin(adata.obs['sample'])]
    cell_counts = cell_counts.merge(total_counts, on='sample')
    cell_counts['percentage'] = cell_counts['count'] / cell_counts['total_count'] * 100  # 转换为百分比
    # 添加 Age_group（作为标识）
    cell_counts = cell_counts.merge(adata.obs[['sample', age_group]].drop_duplicates(), on='sample', how='left')
    cell_counts['Age_group'] = pd.Categorical(cell_counts[age_group], categories=age_order, ordered=True)
    # 按年龄组排序（如果需要）
    cell_counts = cell_counts.sort_values(by='Age_group')
    # 作图
    cell_types = cell_counts[cell_subtype].unique()
    sns.set_style("whitegrid")
    for cell in cell_types:
        plt.figure(figsize=(8, 5))
        data_subset = cell_counts[cell_counts[cell_subtype] == cell]
        # 计算p值和效应量时排除A和H组
        # filtered_data = data_subset[~data_subset['age_group'].isin(['A', 'H'])]
        filtered_data = data_subset
        # Kruskal-Wallis H 检验，计算整体差异的 p 值
        groups = filtered_data['Age_group'].unique()
        group_data = [filtered_data[filtered_data['Age_group'] == group]['percentage'] for group in groups]
        # 计算 p 值
        p_val = stats.kruskal(*group_data)[1]
        # 计算 Eta Squared (η²) for Kruskal-Wallis
        n = sum([len(group) for group in group_data])  # 总样本量
        h_stat = stats.kruskal(*group_data)[0]  # Kruskal-Wallis H 统计量
        eta_squared = h_stat**2 / (h_stat**2 + n - len(groups))  # 计算 Eta Squared
        # 根据效应量和p值确定显著性标记
        if eta_squared < 0.01:
            significance = " (ns)"
        else:
            if p_val <= 0.001:
                significance = "(***)"
            elif p_val <= 0.01:
                significance = "(**)"
            elif p_val <= 0.05:
                significance = "(*)"
            else:
                significance = "(ns)"
        # 绘制 boxplot
        sns.boxplot(
            data=data_subset, x="Age_group", y="percentage", 
            order=age_order, palette="Set2", width=0.6, showfliers=False  # 颜色&箱体宽度&去掉异常值
        )
        # 标注标题（包含显著性标记）
        plt.title(f"{cell}{significance}", fontsize=14, fontweight="bold")
        plt.xticks(fontsize=12)
        plt.xlabel("Age Group", fontsize=12)
        plt.ylabel("Cell Type Percentage", fontsize=12)
        # 获取当前坐标轴的范围
        x_min, x_max = plt.xlim()
        y_min, y_max = plt.ylim()
        # 在图形中间位置添加 p 值和 η²
        plt.text((x_min + x_max) / 2, y_max * 0.95, 
                f"p = {p_val:.3e}, η² = {eta_squared:.3f}", 
                ha='center', fontsize=12, color='black', weight='bold')
        sns.despine()
        fig_path = 'external_inputs/Celltype_general' + Type+'/'+lineage+'/' 
        plt.savefig(
        fig_path+cell +"_agegtrend.pdf",
        dpi=300, bbox_inches='tight',format='pdf')
        plt.savefig(fig_path+cell +"_agegtrend.png", dpi=300, bbox_inches="tight")
        plt.show()


In [ ]:
########### 批量产 Fib & Endo( allAge 或者 NormalAge的 )
from pathlib import Path
base_dir = '/data/project/chenfeiwang/ytwang/'
h5ad_files = list(Path(base_dir).rglob('*age*')) 
h5ad_files = [h5ad_files[i] for i in [0, 1, 3, 4]]


In [ ]:
h5ad_files

In [ ]:
for file in h5ad_files:
    print(file)
    adata = sc.read_h5ad(file)
    lineage = os.path.basename(file).split('_')[0]
    if '_age_normal.h5ad' in str(file):
        Type = 'normal'
    elif '_age.h5ad' in str(file):
        Type = 'all'
    age_order = sorted(adata.obs['Age_group'].unique())
    plot_box_by_celltype_general(
        adata, 
        'Age_group', 
        age_order, 
        'Celltype_general', 
        'Celltype',
        lineage=lineage,
        Type=Type
    )
    plot_box(adata, 'Age_group', age_order, 'Celltype_general',lineage=lineage, Type=Type)
    

In [ ]:
os.path.basename(h5ad_files[0]).split('_')[0]


In [ ]:
os.path.basename(h5ad_files[0]).split('_')[10]
